## Imports

In [41]:
import numpy as np
import pandas as pd
from collections import OrderedDict
import time
from scipy.stats import mode

## Data

In [26]:
X = pd.read_csv("kmeans_data/data.csv", header=None).values 
y = pd.read_csv("kmeans_data/label.csv", header=None).values.ravel() 

In [27]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))
K = len(np.unique(y))
print("Detected K:", K)

X shape: (10000, 784)
y shape: (10000,)
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Detected K: 10


## Distance Functions

In [28]:
def euclidean_distances(X, centroids):
    return np.sqrt(((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))

def cosine_dissimilarity(X, centroids):
    # Compute cosine similarity, then convert to dissimilarity (1 - cos)
    dot = np.dot(X, centroids.T)
    norm_X = np.linalg.norm(X, axis=1, keepdims=True)
    norm_C = np.linalg.norm(centroids, axis=1, keepdims=True).T
    sim = dot / (norm_X * norm_C + 1e-12)
    return 1.0 - sim

def generalized_jaccard_dissimilarity(X, centroids):
    n, d = X.shape
    k = centroids.shape[0]
    dists = np.zeros((n, k))
    for j in range(k):
        a = X
        b = centroids[j]
        numer = np.minimum(a, b).sum(axis=1)
        denom = np.maximum(a, b).sum(axis=1) + 1e-12
        jacc = numer / denom
        dists[:, j] = 1.0 - jacc
    return dists

## Centroids

In [29]:
def init_centroids(X, K, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.choice(X.shape[0], size=K, replace=False)
    return X[idx].astype(float)

## Kmeans

In [30]:
def compute_SSE(X, centroids, labels, dissimilarity='euclidean'):
    # compute SSE using the corresponding dists; SSE uses squared Euclidean by default,
    # but for generality we compute sum of squared dissimilarities (consistent across metrics)
    if dissimilarity == 'euclidean':
        dists = euclidean_distances(X, centroids)
    elif dissimilarity == 'cosine':
        dists = cosine_dissimilarity(X, centroids)
    elif dissimilarity == 'jaccard':
        dists = generalized_jaccard_dissimilarity(X, centroids)
    chosen = dists[np.arange(len(labels)), labels]
    return (chosen**2).sum()

In [31]:
def kmeans(X, K, metric='euclidean', max_iters=500, tol=1e-6, seed=0, verbose=False):
    centroids = init_centroids(X, K, seed)
    sse_history = []
    start = time.time()

    for it in range(1, max_iters + 1):
        # compute dissimilarities
        if metric == 'euclidean':
            dists = euclidean_distances(X, centroids)
        elif metric == 'cosine':
            dists = cosine_dissimilarity(X, centroids)
        elif metric == 'jaccard':
            dists = generalized_jaccard_dissimilarity(X, centroids)
        else:
            raise ValueError("Unknown metric")

        labels = np.argmin(dists, axis=1)
        sse = (dists[np.arange(X.shape[0]), labels] ** 2).sum()
        sse_history.append(sse)

        # update centroids
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            members = X[labels == k]
            if len(members) == 0:
                new_centroids[k] = X[np.random.randint(0, X.shape[0])]
            else:
                new_centroids[k] = members.mean(axis=0)

        shift = np.linalg.norm(new_centroids - centroids)
        if verbose:
            print(f"Iter {it}: SSE={sse:.2f}, Shift={shift:.6f}")

        # stopping conditions
        if shift <= tol:
            stop_reason = "centroid_no_change"
            centroids = new_centroids
            break
        if it > 1 and sse_history[-1] > sse_history[-2]:
            stop_reason = "sse_increase"
            centroids = new_centroids
            break

        centroids = new_centroids
    else:
        stop_reason = "max_iter"

    end = time.time()

    # recompute final labels and SSE
    if metric == 'euclidean':
        dists = euclidean_distances(X, centroids)
    elif metric == 'cosine':
        dists = cosine_dissimilarity(X, centroids)
    else:
        dists = generalized_jaccard_dissimilarity(X, centroids)
    labels = np.argmin(dists, axis=1)
    final_sse = (dists[np.arange(X.shape[0]), labels] ** 2).sum()

    return {
        'centroids': centroids,
        'labels': labels,
        'sse_history': sse_history,
        'iterations': len(sse_history),
        'time_taken': end - start,
        'final_sse': final_sse,
        'stop_reason': stop_reason
    }

## Cluster Labelling

In [32]:
def cluster_majority_labels(labels, y_true, K):
    cluster_labels = np.zeros(K, dtype=int)
    for k in range(K):
        members = y_true[labels == k]
        if len(members) == 0:
            cluster_labels[k] = -1
        else:
            m = mode(members, keepdims=True)
            cluster_labels[k] = m.mode[0]
    return cluster_labels

In [33]:
def clustering_accuracy(labels, y_true, cluster_labels):
    pred = np.array([cluster_labels[l] if cluster_labels[l] != -1 else -1 for l in labels])
    valid = pred != -1
    acc = (pred[valid] == y_true[valid]).sum() / len(y_true[valid])
    return acc

## Results

In [ ]:
results = {}

# Euclidean
res_euc = kmeans(X, K, metric='euclidean', max_iters=500, seed=0)
clabels = cluster_majority_labels(res_euc['labels'], y, K)
acc_euc = clustering_accuracy(res_euc['labels'], y, clabels)
res_euc.update({'accuracy': acc_euc})
results['euclidean'] = res_euc

# Cosine
res_cos = kmeans(X, K, metric='cosine', max_iters=500, seed=0)
clabels = cluster_majority_labels(res_cos['labels'], y, K)
acc_cos = clustering_accuracy(res_cos['labels'], y, clabels)
res_cos.update({'accuracy': acc_cos})
results['cosine'] = res_cos

# Jaccard
res_jac = kmeans(X, K, metric='jaccard', max_iters=500, seed=0)
clabels = cluster_majority_labels(res_jac['labels'], y, K)
acc_jac = clustering_accuracy(res_jac['labels'], y, clabels)
res_jac.update({'accuracy': acc_jac})
results['jaccard'] = res_jac

In [35]:
for name, r in results.items():
    print(f"{name:10s} | SSE: {r['final_sse']:.2f} | Acc: {r['accuracy']:.4f} | "
          f"Iters: {r['iterations']} | Time: {r['time_taken']:.2f}s | Stop: {r['stop_reason']}")

euclidean  | SSE: 25321064456.82 | Acc: 0.5977 | Iters: 48 | Time: 16.84s | Stop: centroid_no_change
cosine     | SSE: 684.09 | Acc: 0.5478 | Iters: 46 | Time: 3.95s | Stop: centroid_no_change
jaccard    | SSE: 3686.35 | Acc: 0.5477 | Iters: 12 | Time: 5.24s | Stop: sse_increase


In [ ]:
def kmeans_forced_stop(X, K, metric='euclidean', stop_type='centroid', max_iters=500, tol=1e-6, seed=0, verbose=False):
    rng = np.random.RandomState(seed)
    centroids = X[rng.choice(X.shape[0], size=K, replace=False)].astype(float)
    sse_history = []
    start = time.time()

    for it in range(1, max_iters+1):
        # compute dissimilarities
        if metric == 'euclidean':
            dists = euclidean_distances(X, centroids)
        elif metric == 'cosine':
            dists = cosine_dissimilarity(X, centroids)
        elif metric == 'jaccard':
            dists = generalized_jaccard_dissimilarity(X, centroids)
        else:
            raise ValueError("Unknown metric")
        labels = np.argmin(dists, axis=1)
        sse = (dists[np.arange(X.shape[0]), labels] ** 2).sum()
        sse_history.append(sse)

        # update centroids
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            members = X[labels == k]
            if len(members) == 0:
                new_centroids[k] = X[np.random.randint(0, X.shape[0])]
            else:
                new_centroids[k] = members.mean(axis=0)

        shift = np.linalg.norm(new_centroids - centroids)

        # apply only the requested stopping rule
        if stop_type == 'centroid' and shift <= tol:
            stop_reason = 'centroid_no_change'
            centroids = new_centroids
            break
        if stop_type == 'sse_increase' and it > 1 and sse > sse_history[-2]:
            stop_reason = 'sse_increase'
            centroids = new_centroids
            break
        if stop_type == 'max_iter' and it == max_iters:
            stop_reason = 'max_iter'
            centroids = new_centroids
            break

        centroids = new_centroids
    else:
        stop_reason = 'max_iter'

    end = time.time()
    # final recompute
    if metric == 'euclidean':
        dists = euclidean_distances(X, centroids)
    elif metric == 'cosine':
        dists = cosine_dissimilarity(X, centroids)
    else:
        dists = generalized_jaccard_dissimilarity(X, centroids)
    labels = np.argmin(dists, axis=1)
    final_sse = (dists[np.arange(X.shape[0]), labels] ** 2).sum()

    return {
        'centroids': centroids,
        'labels': labels,
        'final_sse': final_sse,
        'sse_history': sse_history,
        'iterations': len(sse_history),
        'time_taken': end-start,
        'stop_reason': stop_reason
    }

In [44]:
metrics = ['euclidean', 'cosine', 'jaccard']
stop_types = ['centroid', 'sse_increase', 'max_iter']   
max_iters = 500

summary = []

for metric in metrics:
    for stop in stop_types:
        res = kmeans_forced_stop(X, K, metric=metric, stop_type=stop, max_iters=max_iters, seed=0)
        summary.append(OrderedDict({
            'metric': metric,
            'forced_stop': stop,
            'final_sse': res['final_sse'],
            'iters': res['iterations'],
            'time_s': res['time_taken'],
            'stop_reason': res['stop_reason']
        }))
        print(f"{metric:8s} | stop={stop:11s} | SSE={res['final_sse']:.2f} | iters={res['iterations']:3d} | time={res['time_taken']:.2f}s | reason={res['stop_reason']}")

euclidean | stop=centroid    | SSE=25321064456.82 | iters= 48 | time=17.48s | reason=centroid_no_change
euclidean | stop=sse_increase | SSE=25321064456.82 | iters=500 | time=191.41s | reason=max_iter
euclidean | stop=max_iter    | SSE=25321064456.82 | iters=500 | time=206.75s | reason=max_iter
cosine   | stop=centroid    | SSE=684.09 | iters= 46 | time=4.51s | reason=centroid_no_change
cosine   | stop=sse_increase | SSE=684.09 | iters=500 | time=49.23s | reason=max_iter
cosine   | stop=max_iter    | SSE=684.09 | iters=500 | time=49.82s | reason=max_iter
jaccard  | stop=centroid    | SSE=3690.82 | iters= 55 | time=32.06s | reason=centroid_no_change
jaccard  | stop=sse_increase | SSE=3686.35 | iters= 12 | time=6.66s | reason=sse_increase
jaccard  | stop=max_iter    | SSE=3690.82 | iters=500 | time=296.46s | reason=max_iter


## Questions & Answers

### Q1 Compare the SSEs of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which method is better?

| Metric | Final SSE | Iterations | Stop Reason |
|:--|--:|--:|:--|
| Euclidean | 25,321,064,456.82 | 48 | Centroid no change |
| Cosine | 684.09 | 46 | Centroid no change |
| Jaccard | 3,686.35 | 12 | SSE increase |
  
SSE represents the compactness of clusters (lower = tighter grouping).  
Since each metric computes distance on a different numerical scale, absolute SSE values are not directly comparable.  
Instead, the stability and convergence behavior indicate performance.  

- **Euclidean K-Means** steadily minimized SSE and converged after 48 iterations.  
- **Cosine K-Means** also converged smoothly (46 iterations).  
- **Jaccard K-Means** stopped early due to an SSE increase.  
 
This shows **Euclidean K-Means** achieved the most stable convergence (smooth decrease in SSE), while Jaccard showed earlier termination and possible oscillations. 

### Q2 Compare the accuracies of Euclidean-K-means Cosine-K-means, Jarcard-K-means. First, label each cluster using the majority vote label of the data points in that cluster. Later, compute the predictive accuracy of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which metric is better?

| Metric | Accuracy |
|:--|--:|
| Euclidean | 0.5977 |
| Cosine | 0.5478 |
| Jaccard | 0.5477 |

Predictive accuracy was computed by assigning each cluster the majority ground-truth label.  
Among the three, **Euclidean K-Means** achieved the highest accuracy (≈59.8%), slightly outperforming Cosine and Jaccard (≈54.8%).  
This indicates that Euclidean distance better captured the structure of the data in terms of true labels.  

### Q3 Which method requires more iterations and times to converge?

| Metric | Iterations | Time (s) | Stop Reason |
|:--|--:|--:|:--|
| Euclidean | 48 | 16.84 | Centroid no change |
| Cosine | 46 | 3.95 | Centroid no change |
| Jaccard | 12 | 5.24 | SSE increase |

- **Euclidean** required the most time but reached true convergence (centroid stability).  
- **Cosine** converged quickly with similar iteration counts but less computation time.  
- **Jaccard** terminated early (12 iterations) due to an SSE increase, suggesting instability.  
 
Although slower, **Euclidean K-Means** showed the most reliable convergence; Cosine was faster, and Jaccard was least stable.


### Q4 Compare the SSEs of Euclidean-K-means Cosine-K-means, Jarcard-K-means with respect to the following three terminating conditions: when there is no change in centroid position, when the SSE value increases in the next iteration, when the maximum preset value (e.g., 100) of iteration is complete  
  
| Metric | SSE (Centroid Stop) | SSE (SSE Increase) | SSE (Max Iter = 500) | Observations |
|:--|--:|--:|--:|:--|
| Euclidean | 25,321,064,456.82 | 25,321,064,456.82 | 25,321,064,456.82 | Same SSE across all, stable convergence early |
| Cosine | 684.09 | 684.09 | 684.09 | Stable; SSE increase/max-iter didn’t alter results |
| Jaccard | 3,690.82 | **3,686.35** | 3,690.82 | Early stop (SSE increase) slightly lower SSE due to oscillation |
 
Each stopping condition was tested independently:
- **Centroid stop:** produced the lowest SSE for all metrics (true convergence).  
- **SSE increase:** caused premature halting (especially for Jaccard).  
- **Max iteration:** matched the centroid result for Euclidean and Cosine (they stabilized before 500 iterations).  
 
The **centroid-based stopping rule** is the most reliable and efficient.  
SSE-increase can end the process too early, and the max-iteration rule can be unnecessarily long for already-stable metrics.  

### Q5 What are your summary observations or takeaways based on your algorithmic analysis?

From the experiments:

- **Euclidean K-Means consistently demonstrated the most stable and reliable performance.**  
  It achieved the smoothest decrease in SSE values and the highest clustering accuracy (≈59.8%).  
  Although it required more time to converge, the algorithm always reached a true centroid-stability condition, indicating strong internal consistency and effective minimization of the objective function.

- **Cosine K-Means converged faster but was slightly less accurate.**  
  Its lower computational time came from fewer iterations and simpler distance computations.  
  However, while the clusters were stable, their alignment with labels was weaker than in the Euclidean case.  
  This suggests that cosine dissimilarity captures angular relationships well but does not represent magnitude-based separations effectively for this dataset.

- **Jaccard K-Means exhibited the least stability among the three.**  
  It terminated early due to an increase in SSE and showed oscillatory convergence behavior.  
  Although its accuracy (~54.8%) was comparable to Cosine K-Means, the early stopping implies the algorithm struggled to find stable centroids in high-dimensional continuous data.  
  Jaccard similarity tends to perform better on sparse, non-negative or binary datasets which may not fully apply to this case.

- **Among termination rules, the centroid no-change condition proved the most reliable.**  
  When applied independently, it consistently produced the lowest SSE for all metrics.  
  In contrast, the SSE-increase criterion sometimes triggered prematurely (especially for Jaccard), while the max-iteration condition often ran unnecessarily long after convergence had already been reached.

**In summary**, Euclidean K-Means provided the best overall balance between clustering quality, convergence stability and interpretability.  
Cosine K-Means offered faster runtime but slightly lower accuracy, while Jaccard K-Means was more sensitive to distance fluctuations and benefited least from the chosen stop conditions.  
Therefore for this dataset **Euclidean distance remains the most effective choice for stable and accurate K-Means clustering.** 

## Checks

In [45]:
from sklearn.cluster import KMeans

kmeans_sklearn = KMeans(n_clusters=K, n_init=10, random_state=42)
labels_sklearn = kmeans_sklearn.fit_predict(X)

print("Sklearn inertia (SSE):", kmeans_sklearn.inertia_)

print("Final SSE:", res_euc['final_sse'])


Sklearn inertia (SSE): 25320407927.6064
Final SSE: 25321064456.81865
